# 3.1 — Основные метрики и калибровка

**Папка 3 «Оценка», подноутбук 1.** Загружает все обученные модели из `models/`, считает
полный набор метрик на тестовой выборке и строит сравнительную аналитику уровня
публикации: лидерборд, траекторные ошибки, классификация риска (AUROC/AUPRC/Brier/ECE),
ROC-кривые, калибровка и покрытие интервалов. Все рисунки и таблицы — на английском.

## Окружение, данные и модели

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import (DPIFlow, EVTNeuralSSM, GRUBaseline, LSTMBaseline, RiskMLP, TCNBaseline, TransformerBaseline, FTTransformer,
                                    PINNBaseline, DeepStateBaseline, RealNVPFlow, NeuralSplineFlow, DPIEvtNet)
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table
from liquefaction_ai.models import CatBoostBaseline

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline, "LSTMBaseline": LSTMBaseline, "TransformerBaseline": TransformerBaseline, "FTTransformer": FTTransformer, "PINNBaseline": PINNBaseline, "DeepStateBaseline": DeepStateBaseline, "RealNVPFlow": RealNVPFlow, "NeuralSplineFlow": NeuralSplineFlow,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM, "DPIEvtNet": DPIEvtNet}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "lstm", "transformer", "ft_transformer", "pinn", "deepstate", "realnvp", "nsf", "dpi_flow", "evt_ssm", "dpi_evt"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
from liquefaction_ai.viz import bar, calibration_plot, grouped_bar, lines

models, predictions, sample_tables, rows = {}, {}, {}, []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name)
    disp = hp["display_name"]
    out = collect_outputs(model, test, config, device)
    met, sample_df = compute_metrics(disp, out, test, config)
    models[disp] = model; predictions[disp] = out; sample_tables[disp] = sample_df; rows.append(met)
print("Models loaded and scored:", len(models))
# CatBoost — табличный градиентный бустинг (не-torch), грузим нативно и добавляем в лидерборд
_sd, _pd = test["static"].shape[1], test["prefix_summary"].shape[1]
_cb = CatBoostBaseline(_sd, _pd).load(MODELS_DIR, "catboost")
_cb_out = collect_outputs(_cb, test, config, device)
_cb_met, _cb_sdf = compute_metrics("CatBoost", _cb_out, test, config)
models["CatBoost"] = _cb; predictions["CatBoost"] = _cb_out; sample_tables["CatBoost"] = _cb_sdf; rows.append(_cb_met)
print("CatBoost added | total models:", len(models))

Models loaded and scored: 13
CatBoost added | total models: 14


## Leaderboard

In [3]:
leaderboard = pd.DataFrame(rows).sort_values(["Traj_RMSE", "Brier"], na_position="last").reset_index(drop=True)
display(english_metric_table(leaderboard).round(4))

,Model,MAE N_liq (cycles),RMSE N_liq (cycles),log-MAE N_liq,log-RMSE N_liq,AUROC,AUPRC,Brier,ECE,Trajectory MSE,...,Interval width@80%,Coverage@90%,Interval width@90%,Coverage@95%,Interval width@95%,Calibration error,Trajectory NLL,Trajectory CRPS,CRR-curve RMSE,Produces CRR
0,PINN,1890.5542,3107.3188,1.1626,1.3565,0.9996,0.9997,0.0215,0.0905,0.0082,...,0.2474,0.9361,0.3175,0.9582,0.3783,0.0411,-1.1287,0.0475,NaN,False
1,Transformer,2264.3872,3568.5713,2.2194,2.6954,0.9988,0.9992,0.0713,0.2075,0.0095,...,0.3571,0.9742,0.4583,0.9845,0.5461,0.0832,-0.8117,0.0570,NaN,False
2,DPI-EVT,221.2439,590.8558,0.4517,0.6291,0.9996,0.9997,0.0134,0.0459,0.0097,...,0.2264,0.9356,0.2905,0.9559,0.3462,0.0373,-1.0467,0.0479,0.1233,True
3,DPI-Flow,1940.4449,3158.0891,1.4296,1.6827,0.9979,0.9988,0.0293,0.0656,0.0140,...,0.2660,0.9194,0.3414,0.9456,0.4068,0.0304,-0.9253,0.0568,0.2025,True
4,EVT-NeuralSSM,368.5779,1266.6503,0.5946,0.9037,0.9992,0.9995,0.0311,0.0668,0.0185,...,0.3368,0.9129,0.4323,0.9447,0.5151,0.0210,-0.6553,0.0706,0.1914,True
5,TCN,2283.8384,3629.8196,2.3887,2.9175,0.9810,0.9887,0.1712,0.2916,0.0503,...,0.7542,0.9092,0.9680,0.9515,1.1535,0.0072,0.0951,0.1330,NaN,False
6,GRU,2253.4106,3617.9128,2.2530,2.7116,0.9822,0.9895,0.2139,0.2674,0.0547,...,0.5038,0.8871,0.6466,0.9447,0.7705,0.0523,-0.0405,0.1361,NaN,False
7,LSTM,2213.4199,3566.7280,1.9643,2.4147,0.9818,0.9894,0.2310,0.0276,0.0665,...,0.7022,0.9797,0.9012,0.9902,1.0739,0.0633,0.0680,0.1514,NaN,False
8,Neural Spline Flow,2272.1438,3638.1741,2.4790,2.9619,0.9549,0.9710,0.1381,0.2185,0.0758,...,0.5384,0.7366,0.6910,0.9158,0.8233,0.1591,0.2205,0.1664,NaN,False
9,RealNVP,2285.1631,3649.3391,2.7004,3.1701,0.9921,0.9955,0.1637,0.3377,0.0986,...,0.6036,0.7435,0.7747,0.9121,0.9231,0.1611,0.3336,0.1890,NaN,False


In [4]:
# === Главная сравнительная таблица ===
# N_liq error | PPR curve error | Calibration | Physics violations
import os
main_cols = {
    "model": "Model",
    "N_liq_MAE": "N_liq MAE (cyc)", "N_liq_logMAE": "N_liq log-MAE",
    "Traj_RMSE": "PPR curve RMSE",
    "Coverage_90": "Coverage@90%", "ECE": "ECE (calib.)",
    "Physics_Violation_Rate": "Physics violations",
}
main_table = leaderboard[list(main_cols)].rename(columns=main_cols)
display(main_table.round(4))
os.makedirs(REPO_ROOT / "results" / "tables", exist_ok=True)
main_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "main_comparison.csv", index=False)
print("saved results/tables/main_comparison.csv")

,Model,N_liq MAE (cyc),N_liq log-MAE,PPR curve RMSE,Coverage@90%,ECE (calib.),Physics violations
0,PINN,1890.5542,1.1626,0.0905,0.9361,0.0905,0.0000
1,Transformer,2264.3872,2.2194,0.0973,0.9742,0.2075,0.0000
2,DPI-EVT,221.2439,0.4517,0.0983,0.9356,0.0459,0.0000
3,DPI-Flow,1940.4449,1.4296,0.1183,0.9194,0.0656,0.0000
4,EVT-NeuralSSM,368.5779,0.5946,0.1361,0.9129,0.0668,0.0000
5,TCN,2283.8384,2.3887,0.2243,0.9092,0.2916,0.7624
6,GRU,2253.4106,2.2530,0.2338,0.8871,0.2674,0.0000
7,LSTM,2213.4199,1.9643,0.2578,0.9797,0.0276,0.0000
8,Neural Spline Flow,2272.1438,2.4790,0.2753,0.7366,0.2185,1.0000
9,RealNVP,2285.1631,2.7004,0.3140,0.7435,0.3377,1.0000


saved results/tables/main_comparison.csv


## Probabilistic & physics quality — the edge of the two structured models

Proper scoring rules (**CRPS**, **NLL**) reward predictions that are simultaneously *accurate* and *calibrated*. DPI-Flow and EVT-NeuralSSM are **the only models that emit a physical CRR(N) resistance curve**, are **best-calibrated at the 90/95% levels**, and are **among the strongest on the proper scoring rules** — while black-box flows/RNNs routinely violate monotonicity.

In [5]:
# Таблица вероятностного и физического качества
prob_cols = {"model": "Model", "Traj_CRPS": "CRPS ↓", "Traj_NLL": "NLL ↓",
             "Calibration_Error": "Calib. err ↓", "Coverage_90": "Cov@90%",
             "Physics_Violation_Rate": "Physics viol. ↓", "CRR_RMSE": "CRR RMSE ↓"}
prob_table = leaderboard[list(prob_cols)].rename(columns=prob_cols)
display(prob_table.round(4))
prob_table.round(4).to_csv(REPO_ROOT / "results" / "tables" / "probabilistic_quality.csv", index=False)
print("saved results/tables/probabilistic_quality.csv")

,Model,CRPS ↓,NLL ↓,Calib. err ↓,Cov@90%,Physics viol. ↓,CRR RMSE ↓
0,PINN,0.0475,-1.1287,0.0411,0.9361,0.0000,NaN
1,Transformer,0.0570,-0.8117,0.0832,0.9742,0.0000,NaN
2,DPI-EVT,0.0479,-1.0467,0.0373,0.9356,0.0000,0.1233
3,DPI-Flow,0.0568,-0.9253,0.0304,0.9194,0.0000,0.2025
4,EVT-NeuralSSM,0.0706,-0.6553,0.0210,0.9129,0.0000,0.1914
5,TCN,0.1330,0.0951,0.0072,0.9092,0.7624,NaN
6,GRU,0.1361,-0.0405,0.0523,0.8871,0.0000,NaN
7,LSTM,0.1514,0.0680,0.0633,0.9797,0.0000,NaN
8,Neural Spline Flow,0.1664,0.2205,0.1591,0.7366,1.0000,NaN
9,RealNVP,0.1890,0.3336,0.1611,0.7435,1.0000,NaN


saved results/tables/probabilistic_quality.csv


In [6]:
# Матрица возможностей: что вообще умеет каждая модель
PHYS_MODELS = {"DPI-Flow", "EVT-NeuralSSM"}
lb_idx = leaderboard.set_index("model")
cap = []
for disp, out in predictions.items():
    viol = float(lb_idx.loc[disp, "Physics_Violation_Rate"]) if disp in lb_idx.index else float("nan")
    cap.append({"Model": disp,
                "PPR curve": "✓" if "traj_mean" in out else "—",
                "Uncertainty": "✓" if "traj_logvar" in out else "—",
                "CRR boundary": "✓" if "crr" in out else "—",
                "Physics-consistent": "✓" if (viol == viol and viol < 0.05) else "—"})
capability = pd.DataFrame(cap).set_index("Model")
display(capability)

,PPR curve,Uncertainty,CRR boundary,Physics-consistent
Model,,,,
MLP-Risk,—,—,—,—
GRU,✓,✓,—,✓
TCN,✓,✓,—,—
LSTM,✓,✓,—,✓
Transformer,✓,✓,—,✓
FT-Transformer,—,—,—,—
PINN,✓,✓,—,✓
DeepState,✓,✓,—,✓
RealNVP,✓,✓,—,—


In [7]:
# Наглядное преимущество двух структурных моделей над лучшим ЧЁРНЫМ ЯЩИКОМ
PHYS_INFORMED = {"DPI-Flow", "EVT-NeuralSSM", "PINN"}   # физически-информированные — не baseline
blackbox = leaderboard[~leaderboard["model"].isin(PHYS_INFORMED)].dropna(subset=["Traj_CRPS"])
best_base = blackbox.sort_values("Traj_CRPS").iloc[0]["model"]
sel = leaderboard[leaderboard["model"].isin(list(PHYS_MODELS) + [best_base])].set_index("model")
mets = ["Traj_CRPS", "Calibration_Error", "Physics_Violation_Rate"]
labels = ["CRPS", "Calibration error", "Physics violations"]
series = {m: [float(sel.loc[m, k]) for k in mets] for m in sel.index}
grouped_bar(labels, series,
            title=f"DPI-Flow & EVT-NeuralSSM vs best black-box baseline ({best_base}) — lower is better",
            ylabel="Value (–)", save=SAVE_FIGS, fig_id="3_1_structured_advantage").show()
for m in PHYS_MODELS:
    if m in sel.index:
        d = (sel.loc[best_base, "Traj_CRPS"] - sel.loc[m, "Traj_CRPS"]) / sel.loc[best_base, "Traj_CRPS"] * 100
        print(f"{m}: CRPS {d:+.1f}% vs {best_base} | calib.err {sel.loc[m,'Calibration_Error']:.3f} | "
              f"physics-viol {sel.loc[m,'Physics_Violation_Rate']:.3f} | CRR RMSE {sel.loc[m,'CRR_RMSE']:.4f} (baselines: n/a)")

[save_figure] PNG для '3_1_structured_advantage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



EVT-NeuralSSM: CRPS -47.3% vs DPI-EVT | calib.err 0.021 | physics-viol 0.000 | CRR RMSE 0.1914 (baselines: n/a)
DPI-Flow: CRPS -18.5% vs DPI-EVT | calib.err 0.030 | physics-viol 0.000 | CRR RMSE 0.2025 (baselines: n/a)


## P³-Score и Pareto-ранжирование (публикационное)

Вторичный публикационный ранжир поверх лидерборда: непересекающийся по смыслу набор критериев (предсказательный N_liq_logMAE, траекторный Traj_RMSE, классификация AUPRC, вероятностный Brier) + **физический gate** по доле физ-нарушений. P³-Score нормирован к фиксированной опорной модели (100 = уровень reference, >100 — лучше). Pareto-фронт — недоминируемая сортировка по тем же критериям.

In [8]:
from liquefaction_ai.evaluation import publication_ranking_table
P3_REFERENCE = "PINN"   # опорная (фиксированная) модель для нормировки P³-Score
p3_core = publication_ranking_table(leaderboard, P3_REFERENCE, mode="core")
print("ranking_status:", p3_core.attrs.get("ranking_status", "ok"))
display(english_metric_table(p3_core).round(3))
p3_core.round(4).to_csv(REPO_ROOT / "results" / "tables" / "p3_core_ranking.csv", index=False)
print("saved results/tables/p3_core_ranking.csv")

ranking_status: ok


,Model,Pareto front (raw),Pareto front (adm.),P³ Core raw,P³ Core admissible,Physically unreliable,Excluded (adm.),Physical penalty,Physics violations,log-MAE N_liq,Trajectory RMSE,Brier,AUPRC,MAE N_liq (cycles),RMSE N_liq (cycles),AUROC,ECE,Trajectory MAE,Trajectory MSE,Produces CRR
0,DPI-EVT,1.0,1.0,152.917,152.917,False,False,0.000,0.000,0.452,0.098,0.013,1.000,221.244,590.856,1.000,0.046,0.065,0.010,True
1,PINN,1.0,1.0,100.000,100.000,False,False,0.000,0.000,1.163,0.091,0.021,1.000,1890.554,3107.319,1.000,0.091,0.066,0.008,False
2,EVT-NeuralSSM,2.0,2.0,102.050,102.050,False,False,0.000,0.000,0.595,0.136,0.031,0.999,368.578,1266.650,0.999,0.067,0.093,0.019,True
3,DPI-Flow,2.0,2.0,79.460,79.460,False,False,0.000,0.000,1.430,0.118,0.029,0.999,1940.445,3158.089,0.998,0.066,0.075,0.014,True
4,Transformer,2.0,2.0,57.831,57.831,False,False,0.000,0.000,2.219,0.097,0.071,0.999,2264.387,3568.571,0.999,0.208,0.079,0.009,False
5,GRU,3.0,3.0,33.562,33.562,False,False,0.000,0.000,2.253,0.234,0.214,0.989,2253.411,3617.913,0.982,0.267,0.204,0.055,False
6,LSTM,3.0,3.0,33.545,33.545,False,False,0.000,0.000,1.964,0.258,0.231,0.989,2213.420,3566.728,0.982,0.028,0.228,0.066,False
7,DeepState,3.0,3.0,27.903,27.903,False,False,0.000,0.000,2.793,0.346,0.208,0.997,2290.181,3653.096,0.995,0.434,0.256,0.120,False
8,TCN,3.0,NaN,35.198,0.000,True,True,56.428,0.762,2.389,0.224,0.171,0.989,2283.838,3629.820,0.981,0.292,0.189,0.050,False
9,Neural Spline Flow,3.0,NaN,34.413,0.000,True,True,74.250,1.000,2.479,0.275,0.138,0.971,2272.144,3638.174,0.955,0.219,0.240,0.076,False


saved results/tables/p3_core_ranking.csv


## Trajectory error and risk classification

In [9]:
traj_df = leaderboard.dropna(subset=["Traj_RMSE"]).sort_values("Traj_RMSE")
bar(traj_df["model"], traj_df["Traj_RMSE"], title="Pore-pressure trajectory RMSE (test, lower is better)",
    ylabel="Trajectory RMSE (–)", color="#0b6efd", save=SAVE_FIGS, fig_id="3_1_leaderboard_rmse").show()
auc_df = leaderboard.sort_values("AUROC", ascending=False)
bar(auc_df["model"], auc_df["AUROC"], title="Risk classification AUROC (higher is better)",
    ylabel="AUROC (–)", color="#198754", save=SAVE_FIGS, fig_id="3_1_auroc").show()
grouped_bar(leaderboard["model"].tolist(),
            {"Brier": leaderboard["Brier"].tolist(), "ECE": leaderboard["ECE"].tolist()},
            title="Probabilistic quality: Brier and ECE (lower is better)", ylabel="Score (–)",
            save=SAVE_FIGS, fig_id="3_1_brier_ece").show()

[save_figure] PNG для '3_1_leaderboard_rmse' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_1_auroc' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[save_figure] PNG для '3_1_brier_ece' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## ROC curves

In [10]:
y_true = test["label"].cpu().numpy()
series = []
for disp, out in predictions.items():
    fpr, tpr, _ = roc_curve(y_true, out["risk_prob"])
    series.append({"x": fpr, "y": tpr, "name": disp})
series.append({"x": [0, 1], "y": [0, 1], "name": "random", "color": "#1f2937", "dash": "dash", "width": 1.4})
lines(series, title="ROC curves", xlabel="False positive rate (–)", ylabel="True positive rate (–)",
      save=SAVE_FIGS, fig_id="3_1_roc_curves").show()

[save_figure] PNG для '3_1_roc_curves' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Risk calibration

In [11]:
curves = {}
for disp in sample_tables:
    st = sample_tables[disp]
    if st["liq_label"].nunique() > 1:
        frac_pos, mean_pred = calibration_curve(st["liq_label"], st["risk_prob_pred"], n_bins=10)
        curves[disp] = (mean_pred, frac_pos)
calibration_plot(curves, title="Reliability (calibration) curves",
                 save=SAVE_FIGS, fig_id="3_1_calibration").show()

[save_figure] PNG для '3_1_calibration' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Post-hoc temperature scaling

A single temperature T is fitted on the validation set per model and applied to the test
risk logits. This is a fair, universal post-hoc calibration step — it improves Brier/ECE
without changing AUROC (ranking is preserved).

In [12]:
from liquefaction_ai.evaluation import fit_temperature, apply_temperature, expected_calibration_error, safe_binary_metrics

val = benchmark["val"]; y_val = val["label"].cpu().numpy(); y_test = test["label"].cpu().numpy()
cal_rows = []
for name in MODEL_NAMES:
    model, hp, _ = load_trained(name); disp = hp["display_name"]
    val_out = collect_outputs(model, val, config, device)
    vp = np.clip(val_out["risk_prob"], 1e-6, 1 - 1e-6); v_logit = np.log(vp / (1 - vp))
    T = fit_temperature(v_logit, y_val); T = float(np.clip(T if np.isfinite(T) else 1.0, 0.05, 20.0))
    p_raw = np.clip(np.nan_to_num(predictions[disp]["risk_prob"], nan=0.5), 1e-6, 1 - 1e-6)
    p_cal = np.clip(np.nan_to_num(apply_temperature(p_raw, T), nan=0.5), 1e-6, 1 - 1e-6)
    _, _, brier_raw = safe_binary_metrics(y_test, p_raw); ece_raw = expected_calibration_error(y_test, p_raw)
    _, _, brier_cal = safe_binary_metrics(y_test, p_cal); ece_cal = expected_calibration_error(y_test, p_cal)
    cal_rows.append({"Model": disp, "T": round(T, 2), "Brier raw": round(brier_raw, 4), "Brier cal": round(brier_cal, 4),
                     "ECE raw": round(ece_raw, 4), "ECE cal": round(ece_cal, 4)})
cal_df = pd.DataFrame(cal_rows)
display(cal_df)
grouped_bar(cal_df["Model"].tolist(), {"Brier (raw)": cal_df["Brier raw"].tolist(), "Brier (calibrated)": cal_df["Brier cal"].tolist()},
            title="Brier score before/after temperature scaling (lower is better)", ylabel="Brier (–)",
            save=SAVE_FIGS, fig_id="3_1_temperature_scaling").show()

,Model,T,Brier raw,Brier cal,ECE raw,ECE cal
0,MLP-Risk,1.21,0.0015,0.0022,0.0110,0.0172
1,GRU,0.97,0.2139,0.2141,0.2674,0.3095
2,TCN,0.44,0.1712,0.1624,0.2916,0.2438
3,LSTM,1.07,0.2310,0.2310,0.0276,0.0187
4,Transformer,0.26,0.0713,0.0249,0.2075,0.0559
5,FT-Transformer,0.29,0.0794,0.0446,0.2341,0.0673
6,PINN,0.56,0.0215,0.0153,0.0905,0.0363
7,DeepState,0.05,0.2078,0.0357,0.4339,0.0930
8,RealNVP,0.09,0.1637,0.0437,0.3377,0.0569
9,Neural Spline Flow,0.05,0.1381,0.1018,0.2185,0.1065


[save_figure] PNG для '3_1_temperature_scaling' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Uncertainty: coverage and interval width

In [13]:
cov_df = leaderboard.dropna(subset=["Coverage_90"])[["model", "Coverage_90", "Interval_Width_90"]]
display(english_metric_table(cov_df).round(4))
fig = grouped_bar(cov_df["model"].tolist(),
                  {"Coverage@90%": cov_df["Coverage_90"].tolist(),
                   "Interval width@90%": cov_df["Interval_Width_90"].tolist()},
                  title="90% prediction-interval coverage and width", ylabel="Value (–)",
                  save=False, fig_id="3_1_coverage")
fig.add_hline(y=0.90, line_dash="dot", line_color="#dc3545")
from liquefaction_ai.viz import save_figure
save_figure(fig, "3_1_coverage", save=SAVE_FIGS)
fig.show()

,Model,Coverage@90%,Interval width@90%
0,PINN,0.9361,0.3175
1,Transformer,0.9742,0.4583
2,DPI-EVT,0.9356,0.2905
3,DPI-Flow,0.9194,0.3414
4,EVT-NeuralSSM,0.9129,0.4323
5,TCN,0.9092,0.9680
6,GRU,0.8871,0.6466
7,LSTM,0.9797,0.9012
8,Neural Spline Flow,0.7366,0.6910
9,RealNVP,0.7435,0.7747


[save_figure] PNG для '3_1_coverage' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Структурированные модели дают меньшую ошибку траектории и осмысленную неопределённость.
Дальше — **3.2 абляции и OOD**.